# Prometheus Star: ARC-AGI-3 Solver (Production Edition)
**Architecture:** Bridge v23 (Win-Action Replay)
**Last updated:** 2026-05-02

This notebook is streamlined for solving the official ARC-AGI-3 benchmark using the latest repository updates.

### v17 Changes (Tier 1 + Tier 2)
- **Reward rescaling** — intrinsic curiosity scaled to `0.05 * intrinsic` so it no longer drowns the +1.0 level-completion signal
- **Experience replay fix** — real 4-frame stack snapshots fed to the temporal transformer (was a 4x copy of the current frame, zero velocity)
- **Removed silent macro no-op** — dropped a CRLS mutation block that wasted ~10% of steps on unmapped action strings
- **Auto weight persistence** — `run_live_game()` auto-loads vision weights on entry and saves on exit
- **`ObjectExtractor`** — per-frame background / agent-centroid (via motion) / target-position extraction
- **`GameTypeClassifier`** — first 10 frames classify game into navigation / sorting / pattern / unknown
- **Goal-directed `place` actions** — coordinates dispatched from extracted object positions instead of random clicks
- **Beam search v2** — depth 6 / width 8, `w_extrinsic` decay so the imagination net is down-weighted until it has seen reward, best-sim-coord carried through the beam instead of thrown away



### v22 Changes (nav_dominant detector fixes gravity/physics games)
- **`nav_dominant` signal** — tracks avg pixels changed per move action vs.
  per place action across the entire game session. Physics/gravity games
  (vc33: sprites fall, ~50-100px/move) are reliably distinguished from
  circuit/click games (sk48: ~10px/move). Needs 50 move-action samples.
- **Gravity games keep 15% scanner** — when `avg_move > 30` and
  `avg_move > avg_place × 2`, scanner_prob drops to 15% even after wwr>=4;
  vc33 was solved at window 1 with pure nav then locked out by 60% scanner
  injection — this fix lets nav dominate for the remaining 59 windows.
- **Replaced n_reactive signal** — n_reactive was unreliable: StepCounter/
  timer advances every step so ALL games accumulated n_reactive≥5 quickly,
  mis-classifying nav games (vc33) as scanner games.
- **Refactored step() grid-change block** — compute diff array once per step,
  reuse for argwhere; removes double `np.array()` conversion.

### v21 Changes (adaptive scanner mix based on game reactivity)
- **`n_reactive` signal** — counts place clicks that actually changed the grid;
  games where place never changes the grid (e.g. dc22, ~470 s/window) stay
  nav-dominant (15% scanner probe), saving ~5× time for unresponsive games
- **Threshold lowered 8 → 4** — scanner mode enters after 4 consecutive zero-reward
  windows instead of 8; resumes 4 windows after a win (sk48: solved window 34→~30)
- **60/40 scanner/nav mix for reactive games** — sk48-type circuit games
  get 60% scanner + 40% nav instead of 100% scanner; nav moves help
  wa30/sokoban-style games continue exploring while scanner probes
- **Removed non-navigation family injection** — `GameTypeClassifier` labels all
  games "navigation" (small-motion heuristic); the n_reactive signal is a more
  reliable classifier than the motion-ratio heuristic

### v20 Changes (scanner-primary fallback for unsolved games)
- **`_windows_without_reward` counter** — env tracks consecutive zero-reward windows;
  after ≥ 8 windows with no extrinsic reward NavSolver is bypassed entirely and every
  action is routed through `_GridScanner.next_click()` (systematic place+param sweep)
- **Non-navigation scanner injection** — for games classified as *sorting*, *pattern*,
  or still *unknown*, 30% of NavSolver actions are replaced by scanner clicks so
  place/param combinations are explored even before the 8-window threshold is hit
- **Scanner always reachable** — removed the unreachable `stalled_count > 3` guard;
  scanner is now the final fallback in `solver_action()` for all code paths

### v19 Changes (oscillation fix + target cycling)
- **Removed `place` from NavSolver** — navigation games use move actions only; place was causing
  35%/65% oscillation at every target position, consuming entire windows without progress
- **Removed `< 20px` size filter** — real goal regions may be large; any non-agent non-bg
  object is now a valid target (restores vc33-style wins lost in v18)
- **Target cycling** — after 50 steps on the same target without reward, marks it as
  'failed' and moves to the next candidate object; all marks clear on each new window
- **Fixed oscillation trap** — when agent is exactly at target (dy=dx=0), now uses
  `_random_fallback()` instead of the broken `move_right` default
- **Wall-stall escape threshold lowered** — `_steps_no_progress > 8` (was 15) triggers
  `_random_fallback()` to escape walls faster (beam search fallback removed)
- **Game file inspector cell** added to reveal per-game win conditions


### v18 Changes (NavigationSolver refactor + success direction memory)
- **ObjectExtractor-guided navigation** — NavSolver now uses `ObjectExtractor` small-object detection (< 20 px, non-agent) instead of hardcoded colours {5,2,8} and fixed pixel regions
- **Place-on-target** — when agent is within 2 cells of a target (35% chance) NavSolver returns `place` at the target coordinates instead of a move action
- **`ARC3Action` return type** — `NavigationSolver.next_action()` now returns `ARC3Action` directly; `solver_action()` passes it through without conversion
- **Success direction memory** — after any window with `total_extrinsic > 0`, agent records the dominant move direction and seeds the next window's NavSolver with it (helps vc33 repeat wins)


In [ ]:
# ── Colab / local setup ──────────────────────
import sys, os

REPO_BRANCH = "claude/arc-agi-3-notebook-XVCzt"   # branch carrying the v18 bridge

if 'google.colab' in sys.modules:
    if not os.path.exists('Prometheus_v0_PoC'):
        print(f'Cloning Prometheus repository (branch: {REPO_BRANCH})...')
        os.system(f'git clone -b {REPO_BRANCH} https://github.com/pmcray/Prometheus_v0_PoC.git')
    else:
        os.system(f'git -C Prometheus_v0_PoC fetch origin {REPO_BRANCH}')
        os.system(f'git -C Prometheus_v0_PoC checkout {REPO_BRANCH}')
        os.system(f'git -C Prometheus_v0_PoC pull origin {REPO_BRANCH}')

    print('Installing dependencies...')
    os.system('pip install -q arc-agi scikit-learn')
    os.system('pip install -q -e Prometheus_v0_PoC/')
    sys.path.insert(0, '/content/Prometheus_v0_PoC')
else:
    sys.path.insert(0, '..')

print('Prometheus Star v22.0 (nav_dominant Detector)')
print('Last update: 2026-05-02 12:00 UTC')

import json
import math
import random
import time
import warnings
warnings.filterwarnings('ignore')

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (16, 8)

# ── WP71 imports ───────────
from prometheus.wp71_arc_agi3 import (
    ARC3Action, ARC3Observation, ARC3Episode,
    ARC3WorldModel, ARC3GoalInferrer, ARC3ExplorationPolicy,
    ARC3StrangeLoopAgent, ARC3Benchmark, ARC3BenchmarkReport,
    verify_wp71_exit_criteria, _SyntheticARCGame, _ACTION_TYPES,
)

print('WP71 ARC-AGI-3 module loaded.')
print(f'Canonical action types ({len(_ACTION_TYPES)}): {_ACTION_TYPES}')

### Step 2: API Configuration
To access the full ARC-AGI-3 dataset and benchmark, you need an API key from **[three.arcprize.org](https://three.arcprize.org)**.

1. Log in to [three.arcprize.org](https://three.arcprize.org)
2. Copy your **API Key**
3. In Colab, click the **Secrets** (key icon) on the left sidebar
4. Add a new secret with name `ARC_API_KEY` and paste your key as the value
5. Enable the **Notebook access** toggle for this secret

In [ ]:
# Cell 2: API Configuration & Bridge Loading
import os
try:
    from google.colab import userdata
    ARC_API_KEY = userdata.get("ARC_API_KEY")
    os.environ["ARC_API_KEY"] = ARC_API_KEY
    if ARC_API_KEY:
        print(f"API Key active ({ARC_API_KEY[:6]}...)")
    else:
        print("No API Key found - anonymous access enabled.")
except:
    ARC_API_KEY = None
    print("No API Key found - anonymous access enabled.")

from prometheus.arc3_bridge import *
# Weight auto-load is now handled inside run_live_game() — this call is
# idempotent and only populates the in-memory transformer if a checkpoint exists.
load_vision_weights()
print(f"Bridge v23 loaded (Toolkit Available: {TOOLKIT_AVAILABLE}).")
print("Object-centric modules:",
      "ObjectExtractor" in dir(), "GameTypeClassifier" in dir())

def visualize_arc3_episode(episode, max_steps=10):
    """Visualise the first N steps of an ARC-AGI-3 episode."""
    steps = min(len(episode.history), max_steps)
    if steps == 0: return

    fig, axes = plt.subplots(1, steps, figsize=(2 * steps, 2))
    if steps == 1: axes = [axes]

    for i in range(steps):
        obs, action, reward = episode.history[i]
        grid = np.array(obs.grid)
        axes[i].imshow(grid, cmap='tab20', vmin=0, vmax=15)
        axes[i].set_title(f"S{i}: {action.action_type}\nR={reward:.1f}", fontsize=8)
        axes[i].axis('off')
    plt.tight_layout()
    plt.show()

In [ ]:
# ── Game File Inspector v2 ──────────────────────────────────────────────────
# Extracts the game-class logic (skipping the bulky `sprites = {...}` block).
# Run AFTER Cell 3.
import os, glob, re

env_base = 'environment_files'
if not os.path.exists(env_base):
    env_base = '/content/Prometheus_v0_PoC/environment_files'

game_files = sorted(glob.glob(f'{env_base}/**/*.py', recursive=True))
print(f'Found {len(game_files)} game file(s)\n')

for gf in game_files:
    print('=' * 70)
    print(f'FILE: {gf}')
    print('=' * 70)
    with open(gf) as f:
        content = f.read()

    # 1) Strip the giant sprites = { ... } literal so we can see the logic.
    sprites_match = re.search(r'sprites\s*=\s*\{', content)
    if sprites_match:
        depth, i = 0, sprites_match.end() - 1
        while i < len(content):
            if content[i] == '{': depth += 1
            elif content[i] == '}':
                depth -= 1
                if depth == 0:
                    i += 1; break
            i += 1
        n_sprites = content[sprites_match.start():i].count('Sprite(')
        content_logic = content[:sprites_match.start()] + f'# [SPRITES BLOCK: {n_sprites} sprites omitted]\n' + content[i:]
    else:
        content_logic = content

    # 2) Print the surviving logic (cap at 10000 chars per file).
    if len(content_logic) > 10000:
        print(content_logic[:10000])
        print(f'... [{len(content_logic)-10000} chars truncated]')
    else:
        print(content_logic)

    # 3) Extract win-condition method bodies (step, handle_action, complete, solved).
    win_kws = ['def step(', 'def handle_action(', 'def on_action(', 'def is_solved',
               'def is_complete', 'levels_completed', 'def _check', 'def check_win',
               'def complete(', 'def _win', 'def solve']
    found_any = False
    for kw in win_kws:
        idx = content_logic.find(kw)
        if idx == -1:
            continue
        if not found_any:
            print('\n' + '-'*40 + '  WIN-CONDITION METHODS  ' + '-'*40)
            found_any = True
        # Print up to 1200 chars from the method start
        snippet = content_logic[idx:idx+1200]
        print(f'\n--- {kw} (char {idx}) ---')
        print(snippet)
    print()


In [ ]:
# ── Run Prometheus on ARC-AGI-3 ──────
#
# Bridge v23: nav_dominant detector (avg_move > 30px and > 2× avg_place).
# Place actions that triggered wins are replayed at the start of every window,
# converting one-off lucky discoveries into consistent solutions.

SELECTED_GAMES = ["ls20", "ft09", "vc33"]
if ARC_API_KEY and TOOLKIT_AVAILABLE:
    try:
        arc = arc_agi.Arcade()
        all_envs = arc.get_environments()
        SELECTED_GAMES = [e.game_id for e in all_envs]
        print(f"API Key detected: Loading all {len(SELECTED_GAMES)} games.")
    except:
        print("API Key failed: Falling back to public games.")

N_WINDOWS     = 60
WINDOW_STEPS  = 200

live_results = {}
all_episodes = []
t_start = time.time()

# Run on first 5 games to keep demo snappy if all loaded
GAMES_TO_RUN = SELECTED_GAMES[:5] if len(SELECTED_GAMES) > 3 else SELECTED_GAMES

for game_id in GAMES_TO_RUN:
    print()
    print('=' * 55)
    print(f"  Game: {game_id}")
    print('=' * 55)
    result = run_live_game(game_id=game_id, n_windows=N_WINDOWS, window_steps=WINDOW_STEPS, mutation_rate=0.10, fitness_threshold=0.5, verbose=True)
    if result:
        live_results[game_id] = result
        if result.get('last_episode'):
            all_episodes.append(result['last_episode'])

        sr_val = result.get('solve_rate', 0)
        ms_val = result.get('mean_score', 0)
        fam   = result.get('game_family', 'unknown')
        print(f"  --> solve rate: {sr_val:.0%}  mean score: {ms_val:.3f}  family: {fam}")

        # Visualize the last episode
        if result.get('last_episode'):
            print(f"  Visualising last window of {game_id}...")
            visualize_arc3_episode(result['last_episode'], max_steps=8)

        # Step 5: Persistence (run_live_game already saves, this is belt-and-braces)
        save_vision_weights()

print()
print(f"Total time: {time.time()-t_start:.1f}s")

# Step 3: Latent Space Visualization
if all_episodes:
    print("Plotting Latent Space (PCA projection of Transformer embeddings)...")
    visualize_latent_space(all_episodes)
